In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Nombre_sesión_Colab") \
    .getOrCreate()


# **Show the DF you are working with**

In [ ]:
# When reading Parquet, the schema is included natively.
df = spark.read.format("parquet").load(
    "/content/drive/MyDrive/Mis_Parquet/orders_parquet"
)

df.show()

+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|SalesOrderNumber|SalesOrderLineNumber| OrderDate|     CustomerName|               Email|                Item|Quantity|UnitPrice|     Tax|
+----------------+--------------------+----------+-----------------+--------------------+--------------------+--------+---------+--------+
|         SO43701|                   1|2019-07-01|      Christy Zhu|christy12@adventu...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43704|                   1|2019-07-01|       Julio Ruiz|julio1@adventure-...|Mountain-100 Blac...|       1|  3374.99|269.9992|
|         SO43705|                   1|2019-07-01|        Curtis Lu|curtis9@adventure...|Mountain-100 Silv...|       1|  3399.99|271.9992|
|         SO43700|                   1|2019-07-01|     Ruben Prasad|ruben10@adventure...|  Road-650 Black, 62|       1| 699.0982| 55.9279|
|         SO43703|         

# **Consult the DF using DSL** (DataFrame API)

In [ ]:
# Consult the DF using DSL

from pyspark.sql import functions as F
# Aplicamos las transformaciones y agrupaciones
resultado_df = df.groupBy("CustomerName") \
  .agg(
      F.countDistinct("SalesOrderNumber").alias("#_of_Orders"),
      F.sum(F.col("Quantity") * F.col("UnitPrice")).alias("total purchased")
  ) \
  .orderBy(F.col("total purchased").desc())
# Mostramos usando display() para aprovechar la interfaz interactiva de Fabric
resultado_df.show()  # en Fabric use:   display(resultado_df.limit(10))


+----------------+-----------+----------------+
|    CustomerName|#_of_Orders| total purchased|
+----------------+-----------+----------------+
|   Wyatt Griffin|          1|3578.27001953125|
|     Drew Sharma|          1|3578.27001953125|
|    Claudia Zeng|          1|3578.27001953125|
|      Angela Cox|          1|3578.27001953125|
|       Ivan Rana|          1|3578.27001953125|
|       Peter Jai|          1|3578.27001953125|
|        Tracy Xu|          1|3578.27001953125|
|   Xavier Harris|          1|3578.27001953125|
|       Trisha Li|          1|3578.27001953125|
|  Richard Parker|          1|3578.27001953125|
|     Sharon Nara|          1|3578.27001953125|
| Savannah Morris|          1|3578.27001953125|
| Randall Carlson|          1|3578.27001953125|
|       Alvin Sun|          1|3578.27001953125|
|      Ian Morris|          1|3578.27001953125|
|  Hailey Stewart|          1|3578.27001953125|
|Adriana Gonzalez|          1|3578.27001953125|
|   Christine Pal|          1|3578.27001

# **Query the DataFrame using the SQL engine** (SQL text engine)

In [ ]:
#   Query the DataFrame using the SQL engine (DataFrame API)

# 1. Register the DataFrame as a temporary view
df.createOrReplaceTempView("orders_view")

# 2. Execute a standard SQL query
resulting_sql = spark.sql("""
    SELECT
        CustomerName,
        COUNT(DISTINCT SalesOrderNumber) AS NumeroDeOrdenes,
        SUM(Quantity * UnitPrice) AS TotalComprado
    FROM ventas
    GROUP BY CustomerName
    ORDER BY TotalComprado DESC
""")
# 3. Display the resultin df
resulting_sql.show()   # En fabric use:   display(resultado_sql)




+----------------+---------------+----------------+
|    CustomerName|NumeroDeOrdenes|   TotalComprado|
+----------------+---------------+----------------+
|   Wyatt Griffin|              1|3578.27001953125|
|     Drew Sharma|              1|3578.27001953125|
|    Claudia Zeng|              1|3578.27001953125|
|      Angela Cox|              1|3578.27001953125|
|       Ivan Rana|              1|3578.27001953125|
|       Peter Jai|              1|3578.27001953125|
|        Tracy Xu|              1|3578.27001953125|
|   Xavier Harris|              1|3578.27001953125|
|       Trisha Li|              1|3578.27001953125|
|  Richard Parker|              1|3578.27001953125|
|     Sharon Nara|              1|3578.27001953125|
| Savannah Morris|              1|3578.27001953125|
| Randall Carlson|              1|3578.27001953125|
|       Alvin Sun|              1|3578.27001953125|
|      Ian Morris|              1|3578.27001953125|
|  Hailey Stewart|              1|3578.27001953125|
|Adriana Gon